# [Phase 1] D026 HWP 파일 파싱 및 JSON 변환
**HWP 파싱 실패 2건에 대한 단독 처리 노트북**

---

## 처리 대상
- `D026`, `D043` — HWP 파일, 1차 파이프라인 파싱 실패 문서

## 파싱 전략
```
[A-1] hwp5proc xml  → lxml XML 스트림 파싱  (기존 파이프라인과 동일)
  실패 ↓
[A-2] hwp5proc txt  → 텍스트 전용 파싱     (표 구조 없이 본문만)
  실패 ↓
[B]   LibreOffice HWP → DOCX → python-docx  (Colab: apt install libreoffice)
  실패 ↓
JSON 저장 안 함 / failed_docs.csv에 기록
```

## 산출물 스키마
`D001.json` 등 기존 1차 파이프라인 산출물과 **완전히 동일한** 스키마

## toc 필드
별도 코드로 주입 예정 → 이 노트북에서는 `toc: []` 빈 배열로 저장


## 0.환경 설정 & 라이브러리 설치

In [ ]:
!pip install pyhwp==0.1b15 lxml==6.0.2 'pandas==2.2.2' python-docx -q
print('설치 완료')

In [ ]:
import os, re, json, hashlib, subprocess, tempfile, unicodedata
from pathlib import Path
from datetime import date
from difflib import SequenceMatcher

import pandas as pd
from lxml import etree

print('라이브러리 로드 완료')

## 1.드라이브 마운트 & 경로 설정

> **수정 필요**: 실제 환경에 맞게 경로를 조정하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR   = Path('/content/drive/MyDrive/part3data')
FILES_DIR  = BASE_DIR / 'files'
CSV_PATH   = BASE_DIR / 'data_list.csv'
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'
DOCS_DIR   = OUTPUT_DIR / 'docs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DOC_IDS = ['D026', 'D043']

print(f'출력 경로: {OUTPUT_DIR}')
print(f'처리 대상: {TARGET_DOC_IDS}')

## 2.CSV 전처리 & 대상 파일 매핑

기존 파이프라인과 동일한 전처리를 적용한 뒤 D026·D043 2건만 필터링합니다.

In [ ]:
# ── CSV 로드 ──────────────────────────────────────────────────────
df_all = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df_all.columns = [
    '공고번호','공고차수','사업명','사업금액','발주기관',
    '공개일자','입찰참여시작일','입찰참여마감일',
    '사업요약','파일형식','파일명','텍스트'
]

# ── 한국한의학연구원 제거 (기존 파이프라인과 동일) ───────────────
drop_mask = df_all['파일명'].str.contains('한국한의학연구원', na=False)
df_all = df_all[~drop_mask].reset_index(drop=True)

# ── 해시 중복 제거 ────────────────────────────────────────────────
df_all['_hash'] = df_all['텍스트'].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)
df_all = df_all.drop_duplicates(subset='_hash', keep='first').reset_index(drop=True)

# ── doc_id 부여 (기존 파이프라인과 동일 순서 유지) ───────────────
df_all['doc_id'] = [f'D{str(i+1).zfill(3)}' for i in range(len(df_all))]

# ── 결측치 처리 ───────────────────────────────────────────────────
UNKNOWN = '<unknown>'
str_cols = ['공고번호','공개일자','입찰참여시작일','입찰참여마감일','사업요약']
for col in str_cols:
    df_all[col] = df_all[col].fillna(UNKNOWN).astype(str)
df_all['공고차수'] = df_all['공고차수'].fillna(0).astype(int)
df_all['사업금액'] = df_all['사업금액'].where(df_all['사업금액'].notna(), other=None)

# ── 사업유형 파생 ─────────────────────────────────────────────────
TYPE_KW = [('재구축','재구축'),('고도화','고도화'),('개선','개선'),
           ('개발','개발'),('운영','운영'),('통합','통합'),('구축','구축')]
def infer_type(name):
    for kw, lb in TYPE_KW:
        if kw in str(name): return lb
    return '기타'
df_all['사업유형'] = df_all['사업명'].apply(infer_type)
df_all['재공고여부'] = df_all['공고차수'] > 0

# ── 파일 매핑 (NFC 정규화) ────────────────────────────────────────
def nfc(s):
    return unicodedata.normalize('NFC', str(s).strip())

all_files = list(FILES_DIR.iterdir())
file_map  = {nfc(f.name): f for f in all_files}

def find_file(csv_filename):
    if pd.isna(csv_filename): return None
    return file_map.get(nfc(csv_filename))

df_all['file_path'] = df_all['파일명'].apply(find_file)

# ── D026, D043만 필터링 ───────────────────────────────────────────
df = df_all[df_all['doc_id'].isin(TARGET_DOC_IDS)].reset_index(drop=True)

print(f'전체 문서 수: {len(df_all)}건')
print(f'\n처리 대상 문서:')
print(df[['doc_id','사업명','발주기관','파일형식','file_path']].to_string(index=False))

## 3.공통 유틸리티 함수 정의

기존 파이프라인과 **완전히 동일한** 헬퍼 함수들입니다.


In [ ]:
# ── 헤더 패턴 & 감지 ─────────────────────────────────────────────
HEADER_PATTERNS = [
    (1, re.compile(r'^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVXivx]+[\s\.·]')),
    (1, re.compile(r'^제\s*\d+\s*장\s')),
    (2, re.compile(r'^\d+\.\s+[가-힣A-Za-z]')),
    (2, re.compile(r'^□\s')),
    (3, re.compile(r'^\s{0,4}[가나다라마바사아자차카타파하]\.\s')),
    (3, re.compile(r'^\s{0,4}\d+\)\s')),
]

def detect_header_level(line: str) -> int:
    s = line.strip()
    if len(s) > 80: return 0
    for lv, pat in HEADER_PATTERNS:
        if pat.match(s): return lv
    return 0

# ── 표 유틸리티 ──────────────────────────────────────────────────
def cell_text(cell) -> str:
    return ' '.join(''.join(cell.itertext()).split())

def reconstruct_grid(table_el) -> list:
    rows = int(table_el.get('rows', 1))
    cols = int(table_el.get('cols', 1))
    grid = [['' for _ in range(cols)] for _ in range(rows)]
    for cell in table_el.findall('.//TableCell'):
        r  = int(cell.get('row', 0))
        c  = int(cell.get('col', 0))
        rs = int(cell.get('rowspan', 1))
        cs = int(cell.get('colspan', 1))
        txt = cell_text(cell)
        for rr in range(r, min(r+rs, rows)):
            for cc in range(c, min(c+cs, cols)):
                grid[rr][cc] = txt
    return grid

def classify_table(grid: list) -> str:
    if not grid: return 'wide'
    if len(grid[0]) == 2 and len(grid) >= 2:
        first_col = [r[0] for r in grid if r[0]]
        if first_col and sum(len(v) for v in first_col) / len(first_col) < 20:
            return 'key_value'
    return 'wide'

def grid_to_markdown(grid: list) -> str:
    if not grid: return ''
    cols = len(grid[0])
    lines = ['| ' + ' | '.join(grid[0]) + ' |',
             '| ' + ' | '.join(['---'] * cols) + ' |']
    for row in grid[1:]:
        lines.append('| ' + ' | '.join(row) + ' |')
    return '\n'.join(lines)

def grid_to_kv(grid: list) -> str:
    return ' '.join(
        f"{r[0]}: {r[1]}."
        for r in grid if len(r) >= 2 and r[0]
    )

def serialize_table(grid: list) -> tuple:
    t_type = classify_table(grid)
    if t_type == 'key_value':
        return t_type, grid_to_kv(grid), '세로형 2열 표 → 키-값 문장으로 직렬화'
    return t_type, grid_to_markdown(grid), '가로형 표 → Markdown으로 직렬화'

# ── 장식용 표 판별 ────────────────────────────────────────────────
NUMERIC_PATTERN = re.compile(r'^[\d,\.\-/%원₩()]+$')

def _norm(s): return re.sub(r'\s+', '', str(s))
def _table_plain_text(grid): return ''.join(c for row in grid for c in row if c)
def _text_similarity(a, b):
    if not a or not b: return 0.0
    return SequenceMatcher(None, a, b).ratio()
def _internal_repetition_ratio(grid):
    cells = [_norm(c) for row in grid for c in row if c.strip()]
    if not cells: return 0.0
    return 1 - (len(set(cells)) / len(cells))
def _numeric_cell_ratio(grid):
    cells = [c.strip() for row in grid for c in row if c.strip()]
    if not cells: return 0.0
    return sum(1 for c in cells if NUMERIC_PATTERN.match(c)) / len(cells)

def classify_decorative(grid: list, prev_block: dict) -> tuple:
    n_rows = len(grid)
    n_cols = len(grid[0]) if grid else 0
    if n_rows == 1 and n_cols == 1:
        return True, f'tiny_table(rows={n_rows},cols={n_cols})'
    plain = _norm(_table_plain_text(grid))
    sim = 0.0
    if prev_block is not None and prev_block.get('type') == 'text':
        sim = _text_similarity(plain, _norm(prev_block['content']))
        if sim >= 0.97:
            return True, f'text_overlap={sim:.2f}'
    rep  = _internal_repetition_ratio(grid)
    numr = _numeric_cell_ratio(grid)
    if rep >= 0.5 and numr < 0.1:
        return True, f'internal_repetition={rep:.2f}'
    return False, f'data_table(sim={sim:.2f},rep={rep:.2f},numeric={numr:.2f})'

# ── 섹션 flush ────────────────────────────────────────────────────
def _flush_section(sections, sec_counter, headers, level, blocks):
    """블록이 있을 때만 섹션을 추가하고 sec_counter를 반환"""
    if not blocks:
        return sec_counter
    sec_counter += 1
    sec_id = f'S{str(sec_counter).zfill(2)}'
    sections.append({
        'section_id':       sec_id,
        'header_path':      list(headers),
        'level':            level,
        'blocks':           list(blocks),
        'toc_ref':          None,
        'toc_match_failed': True,
        'block_count':      len(blocks),
        'needs_subsplit':   sum(len(b.get('content', '')) for b in blocks) > 3000,
    })
    return sec_counter

print('공통 유틸리티 정의 완료')

## 4.HWP 파싱 — [A] pyhwp

### A-1: hwp5proc xml (XML 스트림)
기존 파이프라인과 동일. 표의 rowspan/colspan 메타까지 완전 복원.

### A-2: hwp5proc txt (텍스트 전용)
A-1 실패 시 폴백. 표 구조는 잃지만 본문 텍스트는 확보.


In [ ]:
SKIP_CONTROL_TAGS = {'TableControl', 'GShapeObjectControl', 'EqEdit', 'ShapeComponent'}

def paragraph_own_text(para_el) -> str:
    """문단 자체 텍스트만 추출 (표/그림 등 객체 내부 제외)"""
    texts = []
    def walk(el):
        for child in el:
            if child.tag in SKIP_CONTROL_TAGS:
                continue
            if child.tag == 'Text' and child.text:
                texts.append(child.text)
            walk(child)
    walk(para_el)
    return ' '.join(''.join(texts).split())


def _body_items_to_sections(body_items: list, warnings: list) -> tuple:
    """body_items 리스트 → (sections, qa_info)

    body_items 원소:
      ('text',  str)           — 문단 텍스트
      ('table', lxml_element)  — HWP TableBody 요소 (reconstruct_grid 필요)
    """
    sections = []
    current_headers = ['(서두)']
    current_level   = 0
    current_blocks  = []
    sec_counter = blk_counter = text_count = table_count = 0
    wide_count = kv_count = decorative_count = 0

    for itype, ival in body_items:
        if itype == 'text':
            lv = detect_header_level(ival)
            if lv > 0:
                sec_counter = _flush_section(
                    sections, sec_counter, current_headers, current_level, current_blocks
                )
                current_blocks = []
                if lv == 1:
                    current_headers = [ival.strip()]
                elif lv == 2:
                    current_headers = current_headers[:1] + [ival.strip()]
                else:
                    current_headers = current_headers[:2] + [ival.strip()]
                current_level = lv
            else:
                blk_counter += 1
                text_count  += 1
                sec_id = f'S{str(sec_counter + 1).zfill(2)}'
                current_blocks.append({
                    'block_id': f'{sec_id}-B{str(len(current_blocks) + 1).zfill(2)}',
                    'type':    'text',
                    'content': ival,
                })
        else:  # 'table'
            grid = reconstruct_grid(ival)
            if not grid or not any(any(c for c in r) for r in grid):
                warnings.append(f'빈 표 감지 (sec_counter={sec_counter})')
                continue
            t_type, content, note = serialize_table(grid)
            if not content.strip():
                warnings.append(f'표 직렬화 빈 결과 (sec_counter={sec_counter})')
                continue
            blk_counter += 1
            table_count += 1
            wide_count += 1 if t_type == 'wide' else 0
            kv_count   += 1 if t_type == 'key_value' else 0
            prev_block  = current_blocks[-1] if current_blocks else None
            is_dec, dec_reason = classify_decorative(grid, prev_block)
            if is_dec:
                decorative_count += 1
            sec_id = f'S{str(sec_counter + 1).zfill(2)}'
            current_blocks.append({
                'block_id':         f'{sec_id}-B{str(len(current_blocks) + 1).zfill(2)}',
                'type':             'table',
                'table_type':       t_type,
                'content':          content,
                'note':             note,
                'raw_grid':         grid,
                'is_decorative':    is_dec,
                'decorative_reason': dec_reason,
            })

    _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)

    qa = {
        'total_sections':         len(sections),
        'total_blocks':           blk_counter,
        'text_blocks':            text_count,
        'table_blocks':           table_count,
        'table_wide_count':       wide_count,
        'table_key_value_count':  kv_count,
        'decorative_table_count': decorative_count,
        'decorative_table_ratio': round(decorative_count / table_count, 3) if table_count else 0.0,
        'extraction_warnings':    warnings,
    }
    return sections, qa


def parse_hwp_xml(hwp_path: Path) -> tuple:
    """A-1: hwp5proc xml → lxml XML 스트림 파싱
    반환: (sections, qa) 또는 (None, 오류메시지str)
    """
    try:
        result = subprocess.run(
            ['hwp5proc', 'xml', str(hwp_path)],
            capture_output=True, timeout=120,
        )
        if result.returncode != 0:
            err = result.stderr.decode('utf-8', errors='replace')
            return None, f'hwp5proc xml 오류(code={result.returncode}): {err[:300]}'

        # XML 파싱 (불완전 XML은 recover 모드 재시도)
        warnings = []
        try:
            root = etree.fromstring(result.stdout)
        except etree.XMLSyntaxError as e:
            try:
                root = etree.fromstring(result.stdout, etree.XMLParser(recover=True))
                warnings.append(f'XML recover 모드 사용: {e}')
            except Exception as e2:
                return None, f'XML 파싱 실패: {e2}'

        body_items = []
        for el in root.iter():
            if el.tag == 'TableBody':
                body_items.append(('table', el))
            elif el.tag == 'Paragraph':
                if any(a.tag == 'TableCell' for a in el.iterancestors()):
                    continue  # 표 셀 내부 문단 스킵
                txt = paragraph_own_text(el)
                if txt:
                    body_items.append(('text', txt))

        if not body_items:
            return None, 'XML 파싱 성공했으나 body_items 없음'

        sections, qa = _body_items_to_sections(body_items, warnings)
        qa['parse_method'] = 'hwp5xml'
        return (sections, qa), None

    except subprocess.TimeoutExpired:
        return None, 'hwp5proc xml 타임아웃(120s)'
    except Exception as e:
        return None, f'hwp5proc xml 예외: {e}'


def parse_hwp_txt(hwp_path: Path) -> tuple:
    """A-2: hwp5proc txt → 텍스트 전용 파싱 (표 구조 없음)
    반환: (sections, qa) 또는 (None, 오류메시지str)
    """
    try:
        result = subprocess.run(
            ['hwp5proc', 'txt', str(hwp_path)],
            capture_output=True, timeout=120,
        )
        if result.returncode != 0:
            err = result.stderr.decode('utf-8', errors='replace')
            return None, f'hwp5proc txt 오류(code={result.returncode}): {err[:300]}'

        raw_text = result.stdout.decode('utf-8', errors='replace')
        if not raw_text.strip():
            return None, 'hwp5proc txt 결과 없음'

        body_items = [
            ('text', line.strip())
            for line in raw_text.split('\n') if line.strip()
        ]
        warnings = ['hwp5txt 경로 사용 — 표 구조 미복원']
        sections, qa = _body_items_to_sections(body_items, warnings)
        qa['parse_method'] = 'hwp5txt'
        return (sections, qa), None

    except subprocess.TimeoutExpired:
        return None, 'hwp5proc txt 타임아웃(120s)'
    except Exception as e:
        return None, f'hwp5proc txt 예외: {e}'


def parse_hwp_pyhwp(hwp_path: Path):
    """[A] pyhwp 통합 진입점: A-1 실패 시 A-2 폴백
    반환: (sections, qa) 또는 (None, [err1, err2])
    """
    result, err1 = parse_hwp_xml(hwp_path)
    if result is not None:
        print(f'    → A-1 (hwp5xml) 성공')
        return result

    print(f'    → A-1 실패: {err1}')
    result, err2 = parse_hwp_txt(hwp_path)
    if result is not None:
        print(f'    → A-2 (hwp5txt) 성공')
        return result

    print(f'    → A-2 실패: {err2}')
    return None, [err1, err2]

print('pyhwp 파싱 함수 정의 완료')

## 5.HWP 파싱 — [B] LibreOffice 폴백

A-1, A-2가 모두 실패한 경우에만 실행됩니다.
LibreOffice로 HWP → DOCX 변환 후 `python-docx`로 파싱합니다.


In [ ]:
def install_libreoffice() -> bool:
    """LibreOffice 설치 확인 및 설치 (Colab 환경, 최초 1회)"""
    if subprocess.run(['which', 'soffice'], capture_output=True).returncode == 0:
        print('    LibreOffice 이미 설치됨')
        return True
    print('    LibreOffice 설치 중... (수 분 소요)')
    r = subprocess.run(
        ['apt-get', 'install', '-y', 'libreoffice'],
        capture_output=True, timeout=600,
    )
    if r.returncode == 0:
        print('    LibreOffice 설치 완료')
        return True
    print(f'    LibreOffice 설치 실패: {r.stderr.decode()[:200]}')
    return False


def _hwp_to_docx(hwp_path: Path, out_dir: Path) -> Path:
    """soffice CLI로 HWP → DOCX 변환. 변환된 .docx Path 반환"""
    r = subprocess.run(
        ['soffice', '--headless', '--convert-to', 'docx',
         '--outdir', str(out_dir), str(hwp_path)],
        capture_output=True, timeout=180,
    )
    if r.returncode != 0:
        raise RuntimeError(f'soffice 변환 실패: {r.stderr.decode()[:300]}')
    docx_path = out_dir / (hwp_path.stem + '.docx')
    if not docx_path.exists():
        raise FileNotFoundError(f'변환 결과 파일 없음: {docx_path}')
    return docx_path


def _parse_docx_to_sections(docx_path: Path) -> tuple:
    """python-docx로 DOCX 파싱 → (sections, qa)"""
    from docx import Document
    from docx.oxml.ns import qn

    doc      = Document(str(docx_path))
    sections = []
    current_headers = ['(서두)']
    current_level   = 0
    current_blocks  = []
    sec_counter = blk_counter = text_count = table_count = 0
    wide_count = kv_count = decorative_count = 0
    warnings = ['LibreOffice 변환 경로 사용 — 표 병합 셀 근사 복원']

    for element in doc.element.body:
        tag = element.tag.split('}')[-1] if '}' in element.tag else element.tag

        if tag == 'p':
            txt = ''.join(
                node.text for node in element.iter()
                if node.tag.endswith('}t') and node.text
            ).strip()
            if not txt:
                continue
            lv = detect_header_level(txt)
            if lv > 0:
                sec_counter = _flush_section(
                    sections, sec_counter, current_headers, current_level, current_blocks
                )
                current_blocks = []
                if lv == 1:
                    current_headers = [txt.strip()]
                elif lv == 2:
                    current_headers = current_headers[:1] + [txt.strip()]
                else:
                    current_headers = current_headers[:2] + [txt.strip()]
                current_level = lv
            else:
                blk_counter += 1
                text_count  += 1
                sec_id = f'S{str(sec_counter + 1).zfill(2)}'
                current_blocks.append({
                    'block_id': f'{sec_id}-B{str(len(current_blocks) + 1).zfill(2)}',
                    'type':    'text',
                    'content': txt,
                })

        elif tag == 'tbl':
            rows_data = []
            for tr in element.findall('.//' + qn('w:tr')):
                row = [
                    ''.join(
                        n.text for n in tc.iter()
                        if n.tag.endswith('}t') and n.text
                    ).strip()
                    for tc in tr.findall('.//' + qn('w:tc'))
                ]
                if row:
                    rows_data.append(row)

            if not rows_data:
                continue
            max_cols = max(len(r) for r in rows_data)
            grid = [r + [''] * (max_cols - len(r)) for r in rows_data]

            if not any(any(c for c in r) for r in grid):
                warnings.append(f'빈 표 감지 (sec_counter={sec_counter})')
                continue
            t_type, content, note = serialize_table(grid)
            if not content.strip():
                warnings.append(f'표 직렬화 빈 결과 (sec_counter={sec_counter})')
                continue

            blk_counter += 1
            table_count += 1
            wide_count += 1 if t_type == 'wide' else 0
            kv_count   += 1 if t_type == 'key_value' else 0
            prev_block  = current_blocks[-1] if current_blocks else None
            is_dec, dec_reason = classify_decorative(grid, prev_block)
            if is_dec:
                decorative_count += 1
            sec_id = f'S{str(sec_counter + 1).zfill(2)}'
            current_blocks.append({
                'block_id':          f'{sec_id}-B{str(len(current_blocks) + 1).zfill(2)}',
                'type':              'table',
                'table_type':        t_type,
                'content':           content,
                'note':              note,
                'raw_grid':          grid,
                'is_decorative':     is_dec,
                'decorative_reason': dec_reason,
            })

    _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)

    qa = {
        'total_sections':         len(sections),
        'total_blocks':           blk_counter,
        'text_blocks':            text_count,
        'table_blocks':           table_count,
        'table_wide_count':       wide_count,
        'table_key_value_count':  kv_count,
        'decorative_table_count': decorative_count,
        'decorative_table_ratio': round(decorative_count / table_count, 3) if table_count else 0.0,
        'extraction_warnings':    warnings,
        'parse_method':           'libreoffice_docx',
    }
    return sections, qa


def parse_hwp_libreoffice(hwp_path: Path) -> tuple:
    """[B] LibreOffice HWP → DOCX → python-docx 파싱
    반환: (sections, qa) 또는 (None, 오류메시지str)
    """
    with tempfile.TemporaryDirectory() as tmp_dir:
        try:
            docx_path = _hwp_to_docx(hwp_path, Path(tmp_dir))
            sections, qa = _parse_docx_to_sections(docx_path)
            return (sections, qa), None
        except Exception as e:
            return None, f'LibreOffice 파싱 예외: {e}'

print('LibreOffice 폴백 파싱 함수 정의 완료')

## 6.JSON 빌드 함수 정의

기존 파이프라인과 **완전히 동일한** 스키마로 출력합니다.  
`toc` 필드는 별도 코드로 주입 예정이므로 여기서는 `[]`로 저장합니다.


In [ ]:
def build_json(row: pd.Series, sections: list, qa_info: dict) -> dict:
    """sections + qa_info + CSV row → 최종 JSON dict

    toc 필드는 [] 로 고정 (별도 코드로 주입 예정)
    """
    all_text = ' '.join(b['content'] for s in sections for b in s['blocks'])

    # toc가 없으므로 match_rate는 0.0
    qa_info['toc_header_match_rate'] = 0.0
    qa_info['dedup_hash']            = 'sha256:' + hashlib.sha256(all_text.encode()).hexdigest()
    qa_info['needs_subsplit_count']  = sum(1 for s in sections if s.get('needs_subsplit'))

    if qa_info.get('decorative_table_ratio', 0.0) > 0.15:
        qa_info['extraction_warnings'].append(
            f'high_decorative_table_ratio: {qa_info["decorative_table_ratio"]:.1%}'
        )

    return {
        'schema_version': '1.0',
        'doc_id':         row['doc_id'],
        'file_name':      str(row['파일명']),
        'source_format':  str(row['파일형식']).lower(),
        'processed_at':   str(date.today()),
        'metadata': {
            '공고번호':       row['공고번호'],
            '공고차수':       int(row['공고차수']),
            '사업명':         row['사업명'],
            '사업금액':       int(row['사업금액']) if pd.notna(row['사업금액']) else None,
            '발주기관':       row['발주기관'],
            '공개일자':       row['공개일자'],
            '입찰참여시작일': row['입찰참여시작일'],
            '입찰참여마감일': row['입찰참여마감일'],
            '사업요약':       row['사업요약'],
            '사업유형':       row['사업유형'],
            '재공고여부':     bool(row['재공고여부']),
            'linked_doc_id':  None,
            '목차존재':       False,   # toc 주입 후 별도 코드에서 True로 갱신
        },
        'toc':      [],        # 별도 코드로 주입 예정
        'sections': sections,
        'qa':       qa_info,
    }

print('JSON 빌드 함수 정의 완료')

## 7.파이프라인 실행 (D026 & D043)

### 실행 흐름
```
각 문서:
  [A-1] hwp5proc xml 시도
    실패 → [A-2] hwp5proc txt 시도
      실패 → [B] LibreOffice 설치 & 변환 파싱
        실패 → failed_docs 기록, JSON 저장 안 함
```


In [ ]:
manifest_rows = []
failed_docs   = []
_libreoffice_ready = False   # LibreOffice 중복 설치 방지

for _, row in df.iterrows():
    doc_id    = row['doc_id']
    file_path = row['file_path']
    fmt       = str(row['파일형식']).lower()

    print(f'\n[{doc_id}] {str(row["사업명"])[:45]}')

    # ── 파일 존재 확인 ────────────────────────────────────────────
    if file_path is None or not Path(str(file_path)).exists():
        msg = f'파일 없음: {file_path}'
        print(f'  ❌ {msg}')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': msg})
        continue

    if fmt != 'hwp':
        msg = f'예상치 못한 형식: {fmt} (HWP만 처리)'
        print(f'  ⚠️  {msg}')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': msg})
        continue

    hwp_path = Path(str(file_path))
    sections = None
    qa_info  = None

    # ── [A] pyhwp (A-1 → A-2) ────────────────────────────────────
    print(f'  [A] pyhwp 시도...')
    result_a = parse_hwp_pyhwp(hwp_path)
    if result_a[0] is not None:
        sections, qa_info = result_a

    # ── [B] LibreOffice 폴백 ─────────────────────────────────────
    if sections is None:
        print(f'  [B] LibreOffice 폴백 시작...')
        if not _libreoffice_ready:
            _libreoffice_ready = install_libreoffice()
        if _libreoffice_ready:
            result_b, err_b = parse_hwp_libreoffice(hwp_path)
            if result_b is not None:
                sections, qa_info = result_b
                print(f'    → [B] 성공')
            else:
                print(f'    → [B] 실패: {err_b}')
        else:
            print('    → LibreOffice 설치 실패로 폴백 불가')

    # ── 파싱 완전 실패 ────────────────────────────────────────────
    if sections is None or qa_info is None or qa_info.get('total_sections', 0) == 0:
        warnings = (qa_info or {}).get('extraction_warnings', [])
        reason   = warnings[0] if warnings else '알 수 없는 파싱 실패'
        print(f'  ❌ 최종 실패: {reason}')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': reason})
        continue

    # ── JSON 빌드 & 저장 ─────────────────────────────────────────
    doc_json = build_json(row, sections, qa_info)
    out_path = DOCS_DIR / f'{doc_id}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(doc_json, f, ensure_ascii=False, indent=2)

    method   = qa_info.get('parse_method', '?')
    has_warn = len(qa_info['extraction_warnings']) > 0
    print(f'  ✅ 저장 [{method}]  '
          f'섹션={qa_info["total_sections"]}  '
          f'블록={qa_info["total_blocks"]}  '
          f'표={qa_info["table_blocks"]}  '
          f'{"⚠️ " + str(qa_info["extraction_warnings"]) if has_warn else ""}')
    print(f'     → {out_path}')

    manifest_rows.append({
        'doc_id':         doc_id,
        '파일명':          row['파일명'],
        '사업명':          row['사업명'],
        '발주기관':        row['발주기관'],
        '사업유형':        row['사업유형'],
        '사업금액':        row['사업금액'],
        '공개일자':        row['공개일자'],
        '파일형식':        fmt,
        '재공고여부':      row['재공고여부'],
        '목차존재':        False,        # toc 주입 후 갱신 예정
        'total_sections': qa_info['total_sections'],
        'total_blocks':   qa_info['total_blocks'],
        'table_blocks':   qa_info['table_blocks'],
        'has_warning':    has_warn,
        'parse_method':   method,
        'json_path':      str(out_path),
    })

print(f'\n완료: {len(manifest_rows)}건 성공 / {len(failed_docs)}건 실패')

## 8.기존 manifest.csv 업데이트

성공한 문서를 기존 `manifest.csv`에 upsert 합니다.


In [ ]:
if manifest_rows:
    new_df        = pd.DataFrame(manifest_rows)
    manifest_path = OUTPUT_DIR / 'manifest.csv'

    if manifest_path.exists():
        existing = pd.read_csv(manifest_path, encoding='utf-8-sig')
        existing = existing[~existing['doc_id'].isin(TARGET_DOC_IDS)]
        updated  = pd.concat([existing, new_df], ignore_index=True)
        updated.to_csv(manifest_path, index=False, encoding='utf-8-sig')
        print(f'manifest.csv 업데이트: {manifest_path}')
        print(f'  기존 {len(existing)}건 + 신규 {len(new_df)}건 = {len(updated)}건')
    else:
        new_df.to_csv(manifest_path, index=False, encoding='utf-8-sig')
        print(f'manifest.csv 신규 생성: {manifest_path}')

if failed_docs:
    fail_df   = pd.DataFrame(failed_docs)
    fail_path = OUTPUT_DIR / 'failed_docs_d026_d043.csv'
    fail_df.to_csv(fail_path, index=False, encoding='utf-8-sig')
    print(f'\n실패 목록: {fail_path}')
    print(fail_df.to_string(index=False))

## 9.재공고 linked_doc_id 연결

성공한 문서 중 재공고(`공고차수 > 0`)에 대해 원공고 연결을 처리합니다.


In [ ]:
for doc_id in [r['doc_id'] for r in manifest_rows]:
    re_row = df_all[df_all['doc_id'] == doc_id]
    if re_row.empty:
        continue
    re_row = re_row.iloc[0]
    if not re_row['재공고여부']:
        continue

    candidates = df_all[
        (df_all['사업명']   == re_row['사업명'])   &
        (df_all['발주기관'] == re_row['발주기관']) &
        (df_all['공고차수'] <  re_row['공고차수'])
    ]
    if candidates.empty:
        continue
    original_id = candidates.sort_values('공고차수').iloc[0]['doc_id']

    json_path = DOCS_DIR / f'{doc_id}.json'
    if json_path.exists():
        with open(json_path, 'r', encoding='utf-8') as f:
            doc = json.load(f)
        doc['metadata']['linked_doc_id'] = original_id
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(doc, f, ensure_ascii=False, indent=2)
        print(f'재공고 연결: {doc_id} → 원공고 {original_id}')

print('재공고 연결 처리 완료')

## 10.QA 검증

생성된 JSON 파일의 구조와 품질을 확인합니다.


In [ ]:
for doc_id in TARGET_DOC_IDS:
    json_path = DOCS_DIR / f'{doc_id}.json'
    if not json_path.exists():
        print(f'[{doc_id}] ❌ 파일 없음 (파싱 실패)')
        continue

    with open(json_path, 'r', encoding='utf-8') as f:
        doc = json.load(f)

    m, q = doc['metadata'], doc['qa']
    print(f'[{doc_id}] ═══════════════════════════════════════')
    print(f'  사업명      : {m["사업명"]}')
    print(f'  발주기관    : {m["발주기관"]}')
    print(f'  파싱 방법   : {q.get("parse_method", "?")}')
    print(f'  섹션 수     : {q["total_sections"]}')
    print(f'  블록 수     : {q["total_blocks"]}  (표: {q["table_blocks"]})')
    print(f'  경고        : {q["extraction_warnings"]}')
    print()
    for sec in doc['sections'][:3]:
        print(f'  [{sec["section_id"]}] {"/".join(sec["header_path"])}')
        for blk in sec['blocks'][:1]:
            preview = blk['content'][:80].replace('\n', ' ')
            print(f'    [{blk["type"]}] {preview}...')
    print()

In [ ]:
# ── 스키마 호환성 검사: D001과 필드 구조 비교 ───────────────────
print('=== 스키마 호환성 검사 (D001 기준) ===\n')

d001_path = DOCS_DIR / 'D001.json'
if not d001_path.exists():
    print('D001.json 없음 — 비교 건너뜀')
else:
    with open(d001_path, 'r', encoding='utf-8') as f:
        d001 = json.load(f)

    ref = {
        'top':  set(d001.keys()),
        'meta': set(d001['metadata'].keys()),
        'qa':   set(d001['qa'].keys()),
        'sec':  set(d001['sections'][0].keys()) if d001['sections'] else set(),
        'blk':  set(d001['sections'][0]['blocks'][0].keys()) if d001['sections'] else set(),
    }

    for doc_id in TARGET_DOC_IDS:
        path = DOCS_DIR / f'{doc_id}.json'
        if not path.exists():
            print(f'[{doc_id}] 파일 없음')
            continue
        with open(path, 'r', encoding='utf-8') as f:
            doc = json.load(f)

        all_ok = True
        checks = [
            ('top-level',  ref['top'],  set(doc.keys())),
            ('metadata',   ref['meta'], set(doc['metadata'].keys())),
            ('qa',         ref['qa'],   set(doc['qa'].keys())),
        ]
        if doc['sections']:
            checks.append(('section',  ref['sec'], set(doc['sections'][0].keys())))
            if doc['sections'][0]['blocks']:
                checks.append(('block', ref['blk'], set(doc['sections'][0]['blocks'][0].keys())))

        for label, r_keys, t_keys in checks:
            missing = r_keys - t_keys
            extra   = t_keys - r_keys
            if missing:
                print(f'  [{doc_id}] {label} 누락: {missing}')
                all_ok = False
            if extra:
                print(f'  [{doc_id}] {label} 추가 키: {extra}')

        if all_ok:
            print(f'  [{doc_id}] ✅ D001 스키마 완전 호환')
    print()

# [Phase 2]D026 — 목차 병합 및 구조 후처리


---

## 처리 흐름
```
docs/D026.json          (1차 전처리 결과)
toc_docs/D026_toc.json  (수작업 목차)
        │
        ▼
[Phase 1] toc 병합        — 수작업 목차를 JSON에 주입
[Phase 2] header_path 매칭 — 섹션과 목차 항목을 유사도로 연결 (toc_ref)
[Phase 3] 표 중복 정리    — 병합 셀 중복 텍스트 제거
[Phase 4] 대형 섹션 표시  — block_count > 30 에 needs_subsplit 플래그
[Phase 5] qa 재계산 & 저장 → final_data/D026.json
```

## 경로 구조
```
내 드라이브 / part3data / Preprocessed dataset
  ├─ docs/D026.json         ← 입력 ①
  ├─ toc_docs/D026_toc.json ← 입력 ②
  └─ final_data/D026.json   ← 출력
```


## 0.환경 설정 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, re
from pathlib import Path
from difflib import SequenceMatcher
from typing import Any

# ── 경로 (기존 2차 파이프라인과 동일한 체계) ─────────────────────
BASE_DIR   = Path('/content/drive/MyDrive/part3data')
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'

DOCS_DIR  = OUTPUT_DIR / 'docs'        # 입력 ① : 1차 전처리 JSON
TOC_DIR   = OUTPUT_DIR / 'toc_docs'    # 입력 ② : 목차 JSON
FINAL_DIR = OUTPUT_DIR / 'final_data'  # 출력

FINAL_DIR.mkdir(parents=True, exist_ok=True)

# ── 처리 대상 고정 ────────────────────────────────────────────────
TARGET_DOC_ID = 'D026'
DOC_PATH      = DOCS_DIR  / f'{TARGET_DOC_ID}.json'
TOC_PATH      = TOC_DIR   / f'{TARGET_DOC_ID}_toc.json'
OUT_PATH      = FINAL_DIR / f'{TARGET_DOC_ID}.json'

print(f'처리 대상  : {TARGET_DOC_ID}')
print(f'입력 (doc) : {DOC_PATH}')
print(f'입력 (toc) : {TOC_PATH}')
print(f'출력       : {OUT_PATH}')

## 1.입력 파일 로드 & 사전 확인

In [ ]:
# ── 파일 존재 확인 ────────────────────────────────────────────────
assert DOC_PATH.exists(), f'❌ 1차 전처리 결과 없음: {DOC_PATH}'
assert TOC_PATH.exists(), f'❌ 목차 파일 없음: {TOC_PATH}'

with open(DOC_PATH, encoding='utf-8') as f:
    doc = json.load(f)

with open(TOC_PATH, encoding='utf-8') as f:
    toc_data = json.load(f)

# ── doc_id 이중 확인 (파일명 vs 내부 필드) ───────────────────────
assert doc['doc_id'] == TARGET_DOC_ID, \
    f'doc_id 불일치: 파일명={TARGET_DOC_ID}, 내부={doc["doc_id"]}'
assert toc_data['doc_id'] == TARGET_DOC_ID, \
    f'toc doc_id 불일치: 파일명={TARGET_DOC_ID}, 내부={toc_data["doc_id"]}'

print(f'[{TARGET_DOC_ID}] 로드 완료')
print(f'  사업명       : {doc["metadata"]["사업명"]}')
print(f'  섹션 수      : {len(doc["sections"])}')
print(f'  기존 toc     : {len(doc["toc"])}건')
print(f'  새 toc       : {len(toc_data["toc"])}건')
uncertain = sum(1 for t in toc_data['toc'] if t.get('is_uncertain'))
print(f'  is_uncertain : {uncertain}건 (검수 필요)')

##2.toc 병합

수작업으로 만든 `D026_toc.json`의 `toc` 필드를 본문 JSON에 덮어씁니다.


In [ ]:
# ── Phase 1: toc 덮어쓰기 ────────────────────────────────────────
old_toc_count = len(doc['toc'])
doc['toc'] = toc_data['toc']
doc['metadata']['목차존재'] = len(doc['toc']) > 0

print(f'Phase 1 완료')
print(f'  toc: {old_toc_count}건 → {len(doc["toc"])}건')
print(f'  목차 상위 5개:')
for t in doc['toc'][:5]:
    indent = '  ' * (t.get('level', 1) - 1)
    print(f'    {indent}[{t["level"]}] {t.get("number","")} {t["title"]}  (p.{t.get("page","?")})  uncertain={t.get("is_uncertain",False)}')

##3.header_path ↔ toc 매칭

본문 `sections[].header_path` 마지막 텍스트와 목차 항목을 텍스트 유사도로 연결(`toc_ref`)합니다.  
임계값 `threshold=0.45` — 미만이면 `toc_ref: null` + `toc_match_failed: true`.


In [ ]:
def similarity(a: Any, b: Any) -> float:
    a_s = str(a) if a is not None else ''
    b_s = str(b) if b is not None else ''
    return SequenceMatcher(None, a_s, b_s).ratio()


def match_section_to_toc(header_text: str, toc_items: list, threshold: float = 0.45):
    """header_path 마지막 텍스트와 가장 유사한 toc 항목을 반환."""
    best, best_score = None, 0.0
    for item in toc_items:
        toc_title = item.get('title', '') if isinstance(item, dict) else ''
        score = similarity(header_text, toc_title)
        if score > best_score:
            best, best_score = item, score
    if best is not None and best_score >= threshold:
        return best, best_score
    return None, best_score


def attach_toc_ref(doc: dict, threshold: float = 0.45) -> list:
    """모든 섹션에 toc_ref를 부여. 매칭 실패 섹션 목록 반환."""
    failed = []
    toc_items = doc.get('toc', [])
    for section in doc['sections']:
        header_text = str(section['header_path'][-1]) if section.get('header_path') else ''
        matched, score = match_section_to_toc(header_text, toc_items, threshold)
        if matched:
            section['toc_ref'] = {
                'order':       matched['order'],
                'number':      matched['number'],
                'title':       matched['title'],
                'match_score': round(score, 3),
            }
            section['toc_match_failed'] = False
        else:
            section['toc_ref'] = None
            section['toc_match_failed'] = True
            failed.append({
                'section_id':  section['section_id'],
                'header_text': header_text,
                'best_score':  round(score, 3),
            })
    return failed


THRESHOLD = 0.45
failed = attach_toc_ref(doc, threshold=THRESHOLD)

matched_cnt = sum(1 for s in doc['sections'] if not s['toc_match_failed'])
total_cnt   = len(doc['sections'])

print(f'Phase 2 완료  (threshold={THRESHOLD})')
print(f'  전체 섹션     : {total_cnt}건')
print(f'  매칭 성공     : {matched_cnt}건  ({matched_cnt/total_cnt*100:.1f}%)')
print(f'  매칭 실패     : {len(failed)}건')
if failed:
    print(f'  ── 매칭 실패 목록 (best_score 낮은 순) ──')
    for row in sorted(failed, key=lambda r: r['best_score'])[:20]:
        print(f'    [{row["section_id"]}] score={row["best_score"]}  "{row["header_text"][:50]}"')

## 4.표 병합 셀 중복 정리

같은 행에서 바로 앞 칸과 동일한 값이 연속되면 병합 셀 흔적으로 보고 제거합니다.  
정리 후 1행 1열로 줄어든 표는 `is_decorative: true`로 재태깅합니다.


In [ ]:
def dedup_merged_cells(raw_grid: list) -> list:
    """같은 행 안에서 바로 앞 칸과 동일한 값이 연속되면 제거"""
    cleaned = []
    for row in raw_grid:
        new_row = []
        prev = object()   # 첫 칸은 항상 통과시키기 위한 더미 값
        for cell in row:
            if cell == prev:
                continue
            new_row.append(cell)
            prev = cell
        cleaned.append(new_row)
    return cleaned


def grid_to_markdown(grid: list) -> str:
    """정리된 grid를 마크다운 표로 재직렬화"""
    if not grid:
        return ''
    rows = ['| ' + ' | '.join(str(c) for c in row) + ' |' for row in grid]
    sep  = '| ' + ' | '.join(['---'] * len(grid[0])) + ' |'
    return '\n'.join([rows[0], sep] + rows[1:]) if len(rows) > 1 else rows[0] + '\n' + sep


def clean_tables(doc: dict) -> int:
    """문서 내 모든 표 블록을 정리. 변경된 블록 수 반환."""
    changed = 0
    for section in doc['sections']:
        for block in section['blocks']:
            if block.get('type') != 'table':
                continue
            before = block.get('raw_grid', [])
            after  = dedup_merged_cells(before)
            if after != before:
                block['raw_grid'] = after
                block['content']  = grid_to_markdown(after)
                changed += 1
            # 정리 후 1행 1열 이하이면 장식용 표로 재태깅
            flat = [c for row in block['raw_grid'] for c in row]
            if len(block['raw_grid']) <= 1 and len(flat) <= 1 and not block.get('is_decorative'):
                block['is_decorative']    = True
                block['decorative_reason'] = 'dedup_collapsed_to_single_cell'
    return changed


changed = clean_tables(doc)
print(f'Phase 3 완료')
print(f'  정리된 표 블록 : {changed}건')

## 5.대형 섹션 분할 표시

`block_count > BLOCK_THRESHOLD(30)` 인 섹션에 `needs_subsplit: true` 플래그를 부여합니다.


In [ ]:
BLOCK_THRESHOLD = 30

def flag_large_sections(doc: dict, threshold: int = BLOCK_THRESHOLD) -> list:
    flagged = []
    for section in doc['sections']:
        bc = len(section['blocks'])
        section['block_count']    = bc
        section['needs_subsplit'] = bc > threshold
        if section['needs_subsplit']:
            flagged.append({
                'section_id':  section['section_id'],
                'block_count': bc,
                'header_path': section.get('header_path', []),
                'toc_ref':     section.get('toc_ref'),
            })
    return flagged


flagged = flag_large_sections(doc, threshold=BLOCK_THRESHOLD)

print(f'Phase 4 완료  (threshold={BLOCK_THRESHOLD})')
print(f'  needs_subsplit=True 섹션 : {len(flagged)}건')
for row in sorted(flagged, key=lambda r: -r['block_count']):
    print(f'  [{row["section_id"]}] block_count={row["block_count"]}  {row["header_path"]}')

## 6.qa 재계산 & final_data 저장

Phase 1~4 반영 후 품질 지표를 재산출하고 `final_data/D026.json`에 저장합니다.  
`docs/D026.json` 원본은 변경하지 않습니다.


In [ ]:
def recompute_qa(doc: dict) -> dict:
    sections     = doc['sections']
    total_blocks = sum(len(s['blocks']) for s in sections)
    table_blocks = sum(1 for s in sections for b in s['blocks'] if b['type'] == 'table')
    text_blocks  = total_blocks - table_blocks
    decorative   = sum(1 for s in sections for b in s['blocks'] if b.get('is_decorative'))
    matched      = sum(1 for s in sections if not s.get('toc_match_failed'))
    needs_split  = sum(1 for s in sections if s.get('needs_subsplit'))

    qa = doc.get('qa', {})
    qa.update({
        'total_sections':         len(sections),
        'total_blocks':           total_blocks,
        'text_blocks':            text_blocks,
        'table_blocks':           table_blocks,
        'decorative_table_count': decorative,
        'decorative_table_ratio': round(decorative / table_blocks, 3) if table_blocks else 0.0,
        'toc_header_match_rate':  round(matched / len(sections), 3) if sections else 0.0,
        'needs_subsplit_count':   needs_split,
    })
    doc['qa'] = qa
    return qa


qa = recompute_qa(doc)

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Phase 5 완료')
print(f'  저장 경로 : {OUT_PATH}')
print()
print('=== qa 요약 ===')
for k, v in qa.items():
    print(f'  {k:<30} : {v}')

## 7.최종 검증

저장된 `final_data/D026.json`을 다시 로드해서 내용을 확인합니다.


In [ ]:
with open(OUT_PATH, encoding='utf-8') as f:
    check = json.load(f)

print(f'[{check["doc_id"]}] 검증')
print(f'  사업명          : {check["metadata"]["사업명"]}')
print(f'  toc 항목 수     : {len(check["toc"])}')
print(f'  섹션 수         : {len(check["sections"])}')
print(f'  toc 매칭률      : {check["qa"]["toc_header_match_rate"]}')
print(f'  needs_subsplit  : {check["qa"]["needs_subsplit_count"]}건')
print()
print('toc 상위 5개:')
for t in check['toc'][:5]:
    print(f'  {t}')
print()
print('섹션별 toc_ref 상태 (앞 5개):')
for s in check['sections'][:5]:
    print(f'  {s["section_id"]} → toc_ref={s.get("toc_ref")} | needs_subsplit={s.get("needs_subsplit")}')
print()
print('qa:', check['qa'])